# 03 — EDA del dataset CoEdIT

> **Objetivo.** Cargar `grammarly/coedit`, entender su estructura, filtrar por la tarea `gec`, y caracterizar la distribución de longitudes para tomar decisiones de tokenización.

## Sobre CoEdIT

Dataset multitask de **edición de texto** (Raheja et al., 2023). Cada ejemplo trae:

- `src`: texto fuente (a menudo con instrucción inline).
- `tgt`: texto editado.
- `task`: una de `gec`, `paraphrase`, `simplification`, `coherence`, `neutralize`, `formalize`...

La PoC se enfoca en `task == "gec"`.

In [ ]:
# === CONFIG ===
DATASET_NAME = "grammarly/coedit"
N_SAMPLES   = 10_000   # subset para PoC; sube a None para corpus completo
SEED        = 42
# ==============

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_theme(style="whitegrid")

## 1. Cargar el dataset

El loader de Hugging Face cachea automáticamente en `~/.cache/huggingface/datasets/`. Primera ejecución descarga; subsiguientes son instantáneas.

In [ ]:
ds = load_dataset(DATASET_NAME)
print(ds)

In [ ]:
# Primer ejemplo crudo
print(ds["train"][0])

## 2. Distribución de tareas

¿Cuántos ejemplos hay por tipo de tarea? Confirmamos que `gec` está bien representada.

In [ ]:
df_train = pd.DataFrame(ds["train"])
task_counts = df_train["task"].value_counts()
print(task_counts)

fig, ax = plt.subplots(figsize=(8, 4))
task_counts.plot.barh(ax=ax, color="#F06105")
ax.set_xlabel("# ejemplos")
ax.set_title("Distribución de tareas en CoEdIT (train)")
plt.tight_layout()
plt.show()

## 3. Filtrar a tarea GEC

Nos quedamos solo con ejemplos de corrección gramatical. Limpiamos el prefijo de instrucción que CoEdIT incluye en `src` (ej: `"Fix grammar: ..."`) — en la PoC normalizamos a nuestro propio prefijo `"grammar: "` durante preprocesamiento (notebook 04).

In [ ]:
ds_gec = ds.filter(lambda x: x["task"] == "gec")
print("Ejemplos GEC:", {split: len(ds_gec[split]) for split in ds_gec})

In [ ]:
# Inspección manual de 5 ejemplos
for i in range(5):
    ex = ds_gec["train"][i]
    print(f"--- ejemplo {i} ---")
    print("SRC :", ex["src"])
    print("TGT :", ex["tgt"])
    print()

### Limpiar el prefijo de instrucción

CoEdIT codifica la tarea como prefijo de la entrada (`"Fix grammar errors: ..."`). Lo eliminamos para usar nuestro propio prefijo más simple `"grammar: "`.

In [ ]:
import re

INSTRUCTION_PATTERNS = [
    r"^Fix grammar( errors?| in this sentence)?\s*[:.]?\s*",
    r"^Fix the grammar\s*[:.]?\s*",
    r"^Fix grammatical errors in this sentence\s*[:.]?\s*",
    r"^Remove grammatical errors\s*[:.]?\s*",
    r"^Improve the grammar of this text\s*[:.]?\s*",
    r"^Make this sentence grammatical\s*[:.]?\s*",
    r"^Correct the grammar of\s*[:.]?\s*",
    r"^Make the sentence grammatical\s*[:.]?\s*",
    r"^Grammar improvements\s*[:.]?\s*",
    r"^Update to use correct grammar\s*[:.]?\s*",
]

def strip_instruction(text: str) -> str:
    for pat in INSTRUCTION_PATTERNS:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    return text.strip()

def clean_example(ex):
    return {"src": strip_instruction(ex["src"]), "tgt": ex["tgt"], "task": ex["task"]}

ds_gec = ds_gec.map(clean_example)

# Verificar limpieza
for i in range(3):
    print("SRC limpio:", ds_gec["train"][i]["src"])
    print("TGT       :", ds_gec["train"][i]["tgt"])
    print()

## 4. Distribución de longitudes

Decisión clave de tokenización: ¿qué `max_length` usar?

- Si es muy corto → truncamos ejemplos válidos.
- Si es muy largo → desperdiciamos compute en padding.

In [ ]:
df_gec = pd.DataFrame(ds_gec["train"])
df_gec["src_words"] = df_gec["src"].str.split().map(len)
df_gec["tgt_words"] = df_gec["tgt"].str.split().map(len)
df_gec["delta"]    = df_gec["tgt_words"] - df_gec["src_words"]

print(df_gec[["src_words", "tgt_words", "delta"]].describe(percentiles=[.5, .9, .95, .99]).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df_gec["src_words"], bins=50, ax=axes[0], color="#F06105")
axes[0].axvline(df_gec["src_words"].quantile(0.95), color="black", ls="--", label="p95")
axes[0].set_title("Longitud de SRC (palabras)")
axes[0].legend()

sns.histplot(df_gec["delta"], bins=40, ax=axes[1], color="#3578A0")
axes[1].set_title("Δ longitud (tgt - src)")
axes[1].set_xlabel("Δ palabras")

plt.tight_layout()
plt.show()

**Lectura.** El p95 de longitud en palabras suele ser ~30-40. Tras tokenizar con SentencePiece (subwords) cada palabra ≈ 1.3 tokens promedio, así que `max_length=128` cubre cómodamente la cola larga. Validaremos esto en notebook 04 con el tokenizer real.

## 5. Cuántos ejemplos cambian realmente vs `src == tgt`

Un ejemplo donde `src == tgt` significa "oración ya correcta, no editar". Útiles pero no enseñan corrección — la PoC los conserva porque T5 también debe aprender a **no inventar errores**.

In [ ]:
n_total = len(df_gec)
n_equal = (df_gec["src"] == df_gec["tgt"]).sum()
print(f"Ejemplos donde src == tgt: {n_equal}/{n_total} ({n_equal/n_total:.1%})")

## 6. Construir splits PoC

Tomamos `N_SAMPLES` aleatorios y hacemos un 90/10 train/eval. El notebook 04 consume estos splits.

In [ ]:
ds_gec_train = ds_gec["train"].shuffle(seed=SEED)
if N_SAMPLES is not None:
    ds_gec_train = ds_gec_train.select(range(min(N_SAMPLES, len(ds_gec_train))))

split = ds_gec_train.train_test_split(test_size=0.1, seed=SEED)
print("Train:", len(split["train"]), "  Eval:", len(split["test"]))

In [ ]:
# Guardar splits filtrados a disco para que el notebook 04 no repita filter+clean
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT  = ROOT / "artifacts" / "coedit_gec_splits"
OUT.parent.mkdir(exist_ok=True)
split.save_to_disk(str(OUT))
print("Guardado en:", OUT)

## Resumen

- Dataset: **CoEdIT** filtrado a `task == 'gec'`.
- Limpieza: prefijos de instrucción removidos.
- Tamaño PoC: 10 000 ejemplos → 9 000 train / 1 000 eval.
- Longitud SRC p95 ≈ 30-40 palabras → `max_length=128` tokens es seguro.

**Siguiente → `04_preprocesamiento.ipynb`**
